# Regressions on monthly data


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.families import NegativeBinomial



In [ ]:
df = pd.read_csv('../data/intermediate/monthly.csv')
df = df.drop(columns=["Unnamed: 0"])

count_vars = [
    "total", "pmc", "prisoners", "drafted", "volunteers", "contract",
    "slavic_name", "non_slavic_name"
]

df[count_vars] = df[count_vars].fillna(0)
df['post'] = (df['Date'] >= '2022-09').astype(int)

df["total_per100k"] = df["total"]/df['population']*100_000
df["pmc_per100k"] = df["pmc"]/df['population']*100_000
df["prisoners_per100k"] = df["prisoners"]/df['population']*100_000
df["drafted_per100k"] = df["drafted"]/df['population']*100_000
df["volunteers_per100k"] = df["volunteers"]/df['population']*100_000
df["contract_per100k"] = df["contract"]/df['population']*100_000
df['nup'] = df['post'] * df['ex_nup_per100k']

df["slavic_per100k"] = df["slavic_name"]/df['population']*100_000
df["non_slavic_per100k"] = df["non_slavic_name"]/df['population']*100_000


## Total

In [23]:
df_model = df[['Region','Date','total_per100k', 'post', 'ex_nup_per100k', 'nup', 'share_poverty', '% Russians',
               'median_income', 'uneployment', 'urban_share', 'population',
               'insider', 'nat_rep']].copy()
df_model['post_x_rep'] = df['post'] * df['nat_rep']
df_model['post_x_ins'] = df['post'] * df['insider']
df_model['post_x_pov'] = df['post'] * df['share_poverty']

df_model['nup_x_rep'] = df['nup'] * df['nat_rep']
df_model['nup_x_ins'] = df['nup'] * df['insider']
df_model['nup_x_pov'] = df['nup'] * df['share_poverty']

df_model = df_model.dropna()  
df_model

,Region,Date,total_per100k,post,ex_nup_per100k,nup,share_poverty,% Russians,median_income,uneployment,urban_share,population,insider,nat_rep,post_x_rep,post_x_ins,post_x_pov,nup_x_rep,nup_x_ins,nup_x_pov
0,Алтайский край,2022-02-01,0.046406,0,343.285700,0.000000,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
1,Алтайский край,2022-03-01,1.624205,0,343.285700,0.000000,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
2,Алтайский край,2022-04-01,1.484988,0,343.285700,0.000000,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
3,Алтайский край,2022-05-01,2.877164,0,343.285700,0.000000,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
4,Алтайский край,2022-06-01,1.577799,0,343.285700,0.000000,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3550,Ярославская область,2025-03-01,0.414731,1,232.629579,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,1.0,8.8,0.0,232.629579,2047.140297
3551,Ярославская область,2025-04-01,0.000000,1,232.629579,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,1.0,8.8,0.0,232.629579,2047.140297
3552,Ярославская область,2025-05-01,0.000000,1,232.629579,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,1.0,8.8,0.0,232.629579,2047.140297
3553,Ярославская область,2025-06-01,0.000000,1,232.629579,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,1.0,8.8,0.0,232.629579,2047.140297


In [ ]:
nb_model_b_rep = smf.glm(
    formula="total_per100k ~ nat_rep + post + post_x_rep + share_poverty + insider",
    data=df_model,
    family=NegativeBinomial()).fit()
nb_model_nup_rep = smf.glm(
    formula="total_per100k ~ nat_rep + nup + nup_x_rep + share_poverty + insider",
    data=df_model,
    family=NegativeBinomial()).fit()
nb_model_b_ins = smf.glm(
    formula="total_per100k ~ post + insider + post_x_ins + share_poverty + nat_rep",
    data=df_model,
    family=NegativeBinomial()).fit()
nb_model_nup_ins = smf.glm(
    formula="total_per100k ~ nup + insider + nup_x_ins + share_poverty + nat_rep",
    data=df_model,
    family=NegativeBinomial()).fit()
nb_model_b_pov = smf.glm(
    formula="total_per100k ~ post + share_poverty + post_x_pov + insider + nat_rep ",
    data=df_model,
    family=NegativeBinomial()).fit()
nb_model_nup_pov = smf.glm(
    formula="total_per100k ~ nup + share_poverty + nup_x_pov + insider + nat_rep ",
    data=df_model,
    family=NegativeBinomial()).fit()


c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value

## Conscripts

In [27]:
df_model = df[['Region','Date','drafted_per100k', 'post', 'ex_nup_per100k', 'nup', 'share_poverty', '% Russians',
               'median_income', 'uneployment', 'urban_share', 'population',
               'insider', 'nat_rep']].copy()
df_model['post_x_rep'] = df['post'] * df['nat_rep']
df_model['post_x_ins'] = df['post'] * df['insider']
df_model['post_x_pov'] = df['post'] * df['share_poverty']

df_model['nup_x_rep'] = df['nup'] * df['nat_rep']
df_model['nup_x_ins'] = df['nup'] * df['insider']
df_model['nup_x_pov'] = df['nup'] * df['share_poverty']

df_model = df_model.dropna()  
df_model

,Region,Date,drafted_per100k,post,ex_nup_per100k,nup,share_poverty,% Russians,median_income,uneployment,urban_share,population,insider,nat_rep,post_x_rep,post_x_ins,post_x_pov,nup_x_rep,nup_x_ins,nup_x_pov
0,Алтайский край,2022-02-01,0.0,0,343.285700,0.000000,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
1,Алтайский край,2022-03-01,0.0,0,343.285700,0.000000,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
2,Алтайский край,2022-04-01,0.0,0,343.285700,0.000000,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
3,Алтайский край,2022-05-01,0.0,0,343.285700,0.000000,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
4,Алтайский край,2022-06-01,0.0,0,343.285700,0.000000,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3550,Ярославская область,2025-03-01,0.0,1,232.629579,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,1.0,8.8,0.0,232.629579,2047.140297
3551,Ярославская область,2025-04-01,0.0,1,232.629579,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,1.0,8.8,0.0,232.629579,2047.140297
3552,Ярославская область,2025-05-01,0.0,1,232.629579,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,1.0,8.8,0.0,232.629579,2047.140297
3553,Ярославская область,2025-06-01,0.0,1,232.629579,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,1.0,8.8,0.0,232.629579,2047.140297


In [37]:
draft_b_rep = smf.glm(
    formula="drafted_per100k ~ nat_rep + post + post_x_rep + share_poverty + insider",
    data=df_model,
    family=NegativeBinomial()).fit()
draft_nup_rep = smf.glm(
    formula="drafted_per100k ~ nat_rep + nup + nup_x_rep + share_poverty + insider + C(Date)",
    data=df_model,
    family=NegativeBinomial()).fit()
draft_b_ins = smf.glm(
    formula="drafted_per100k ~ post + insider + post_x_ins + share_poverty + nat_rep",
    data=df_model,
    family=NegativeBinomial()).fit()
draft_nup_ins = smf.glm(
    formula="drafted_per100k ~ nup + insider + nup_x_ins + share_poverty + nat_rep + C(Date)",
    data=df_model,
    family=NegativeBinomial()).fit()
draft_b_pov = smf.glm(
    formula="drafted_per100k ~ post + share_poverty + post_x_pov + insider + nat_rep ",
    data=df_model,
    family=NegativeBinomial()).fit()
draft_nup_pov = smf.glm(
    formula="drafted_per100k ~ nup + share_poverty + nup_x_pov + insider + nat_rep  + C(Date)",
    data=df_model,
    family=NegativeBinomial()).fit()


c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value

In [38]:
print(draft_nup_pov.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:        drafted_per100k   No. Observations:                 3555
Model:                            GLM   Df Residuals:                     3508
Model Family:        NegativeBinomial   Df Model:                           46
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1866.0
Date:                 Ср, 06 авг 2025   Deviance:                       1500.3
Time:                        00:17:16   Pearson chi2:                 2.95e+03
No. Iterations:                    24   Pseudo R-squ. (CS):             0.1725
Covariance Type:            nonrobust                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept               -25.75

## Raw count data: slavic

In [96]:
df_model = df[['Region', 'Date','slavic_per100k', 'post', 'ex_nup_per100k', 'share_poverty', '% Russians',
               'median_income', 'uneployment', 'urban_share', 'population',
               'insider', 'nat_rep']].copy()
df_model['post_x_rep'] = df['post'] * df['nat_rep']
df_model['post_x_pov'] = df['post'] * df['share_poverty']
df_model['post_x_ins'] = df['post'] * df['insider']
df_model = df_model.dropna()  
df_model

,Region,Date,slavic_per100k,post,ex_nup_per100k,share_poverty,% Russians,median_income,uneployment,urban_share,population,insider,nat_rep,post_x_rep,post_x_pov,post_x_ins
0,Алтайский край,2022-02-01,0.046406,0,343.285700,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0
1,Алтайский край,2022-03-01,1.531394,0,343.285700,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0
2,Алтайский край,2022-04-01,1.392176,0,343.285700,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0
3,Алтайский край,2022-05-01,2.645134,0,343.285700,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0
4,Алтайский край,2022-06-01,1.438582,0,343.285700,15.4,95.45,24037.0,3.7,0.582394,2154900.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3550,Ярославская область,2025-03-01,0.331785,1,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,8.8,1.0
3551,Ярославская область,2025-04-01,0.000000,1,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,8.8,1.0
3552,Ярославская область,2025-05-01,0.000000,1,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,8.8,1.0
3553,Ярославская область,2025-06-01,0.000000,1,232.629579,8.8,96.52,30524.1,5.0,0.810717,1205600.0,1.0,0.0,0.0,8.8,1.0


In [93]:
nb_model = smf.glm(
    formula="slavic_per100k ~ post + nat_rep + post_x_rep",
    data=df_model,
    family=NegativeBinomial()).fit()

print(nb_model.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:         slavic_per100k   No. Observations:                 3555
Model:                            GLM   Df Residuals:                     3551
Model Family:        NegativeBinomial   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -6259.3
Date:                 Вт, 05 авг 2025   Deviance:                       3100.1
Time:                        21:19:04   Pearson chi2:                 3.87e+03
No. Iterations:                     5   Pseudo R-squ. (CS):            0.04277
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.1894      0.074     -2.568      0.0

c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [94]:
nb_model = smf.glm(
    formula="slavic_per100k ~ post + insider + post_x_ins",
    data=df_model,
    family=NegativeBinomial()).fit()

print(nb_model.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:         slavic_per100k   No. Observations:                 3555
Model:                            GLM   Df Residuals:                     3551
Model Family:        NegativeBinomial   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -6253.2
Date:                 Вт, 05 авг 2025   Deviance:                       3087.9
Time:                        21:19:33   Pearson chi2:                 3.99e+03
No. Iterations:                     5   Pseudo R-squ. (CS):            0.04605
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.1244      0.086     -1.451      0.1

c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [97]:
nb_model = smf.glm(
    formula="slavic_per100k ~ post + share_poverty + post_x_pov",
    data=df_model,
    family=NegativeBinomial()).fit()

print(nb_model.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:         slavic_per100k   No. Observations:                 3555
Model:                            GLM   Df Residuals:                     3551
Model Family:        NegativeBinomial   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -6258.7
Date:                 Вт, 05 авг 2025   Deviance:                       3098.9
Time:                        21:20:28   Pearson chi2:                 4.00e+03
No. Iterations:                     8   Pseudo R-squ. (CS):            0.04309
Covariance Type:            nonrobust                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -0.4455      0.169     -2.630

c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [59]:
nb_model = smf.glm(
    formula="non_slavic_per100k ~ ex_nup_per100k + share_poverty + nat_rep  + insider",
    data=df_model,
    family=NegativeBinomial()).fit()

print(nb_model.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:     non_slavic_per100k   No. Observations:                 3555
Model:                            GLM   Df Residuals:                     3550
Model Family:        NegativeBinomial   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2554.4
Date:                 Вт, 05 авг 2025   Deviance:                       1561.4
Time:                        17:43:20   Pearson chi2:                 2.24e+03
No. Iterations:                    10   Pseudo R-squ. (CS):             0.2018
Covariance Type:            nonrobust                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -2.7544      0.103    -26.

c:\Users\Robert_Beta\miniforge3\envs\epp\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
